# Evaluación con modelo real: Colab / Kaggle
Mide modelos dentro de `centro-ia-agentic-harness` y compara los reportes.

**Estado:** este notebook **no fue ejecutado** por quien lo escribió (sin GPU en su entorno). Si una celda falla, el error es dato útil: anotalo.

Antes: el repo tiene que estar en GitHub (con la v0.3.0). Activá GPU (Colab: Entorno de ejecución > T4; Kaggle: Accelerator > GPU T4).

In [ ]:
REPO = "https://github.com/fleonmoloney/centro-ia-agentic-harness"
import os, sys, subprocess
ENTORNO = "kaggle" if os.path.exists("/kaggle") else ("colab" if "google.colab" in sys.modules else "otro")
print("Entorno:", ENTORNO)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout or "Sin GPU")

In [ ]:
!git clone -q $REPO repo
%cd repo
!pip install -q -e ".[dev,persist,local]"
!git log --oneline | head -3

## 1. Pruebas y evals deterministas (no usan modelo)

In [ ]:
!python -m pytest -q
!python -m centro_harness.evals --cases evals/cases.jsonl

## 2. Modelos locales con `transformers`
Cada modelo corre en un **subproceso** para liberar la memoria de GPU entre uno y otro. Una T4 (16 GB) aloja modelos chicos; no cargues uno de ~30B.
`--max-tokens` limita lo generado por llamada (con modelos de razonamiento, subilo).

In [ ]:
MODELOS = [
    ("Qwen/Qwen2.5-3B-Instruct", "4bit"),
    ("Qwen/Qwen2.5-3B-Instruct", "fp16"),
]
# v1 = esquema JSON completo (línea base); v2 = objeto de ejemplo. Los resultados NO son comparables entre sí.
ESTILOS = ["schema", "example"]
os.makedirs("reports", exist_ok=True)
for modelo, cuant in MODELOS:
  for estilo in ESTILOS:
    nombre = modelo.split("/")[-1] + "_" + cuant + "_" + estilo
    cmd = [sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
           "--llm", "transformers", "--model", modelo, "--quantization", cuant,
           "--max-tokens", "512", "--repeats", "1", "--prompt-style", estilo,
           "--label", f"{modelo} {cuant} prompt-{estilo} ({ENTORNO})",
           "--out-json", f"reports/{nombre}.json"]
    print("\n>>>", " ".join(cmd))
    r = subprocess.run(cmd)
    print("código de salida:", r.returncode)  # 0 ok, 1 fallos, 2 invariante/infra, 3 cortada

## 3. Un proveedor por API (opcional)
Usá **solo datos sintéticos** (los casos del repo lo son). Leé `docs/providers.md` para cuotas: una pasada usa 31+ llamadas.

In [ ]:
import json, urllib.request
# Los IDs gratuitos cambian sin aviso: se consulta el catálogo vivo (no requiere clave).
catalogo = json.load(urllib.request.urlopen("https://openrouter.ai/api/v1/models", timeout=30))["data"]
gratis = sorted(m["id"] for m in catalogo if m["id"].endswith(":free"))
print(len(gratis), "modelos gratuitos ahora mismo:")
print("\n".join(gratis))

In [ ]:
from getpass import getpass
os.environ["HARNESS_API_KEY"] = getpass("API key (no se guarda en el notebook): ")
BASE_URL = "https://openrouter.ai/api/v1"
MODELO_API = ""   # <-- copiá acá UN id de la lista de arriba (p. ej. uno de nvidia/, qwen/ o z-ai/)
assert MODELO_API in gratis, "Elegí un id exacto de la lista"
# Prueba barata antes de gastar la cuota: un solo caso.
cmd = [sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
       "--llm", "compat", "--base-url", BASE_URL, "--model", MODELO_API, "--only", "salud-urgente",
       "--max-tokens", "4096", "--min-interval", "3.1", "--label", f"openrouter/{MODELO_API} prueba"]
print("código de salida:", subprocess.run(cmd).returncode)

In [ ]:
# Pasada completa (31+ llamadas; ver docs/providers.md por la cuota diaria).
cmd = [sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
       "--llm", "compat", "--base-url", BASE_URL, "--model", MODELO_API,
       "--repeats", "1", "--max-tokens", "4096", "--min-interval", "3.1", "--max-total-calls", "45",
       "--prompt-style", "example", "--label", f"openrouter/{MODELO_API} prompt-example",
       "--out-json", "reports/api_openrouter_example.json"]
print("código de salida:", subprocess.run(cmd).returncode)

## 4. Comparar y descargar

In [ ]:
!python -m centro_harness.compare reports/*.json --out-md reports/comparacion.md
print(open("reports/comparacion.md", encoding="utf-8").read())

In [ ]:
!zip -qr reports.zip reports
try:
    from google.colab import files; files.download("reports.zip")
except ImportError:
    print("En Kaggle: reports.zip queda en /kaggle/working (pestaña Output).")

Pegá `reports/comparacion.md` en el README del repo **con el modelo, la cuantización, la fecha y el entorno**. Sin eso el número no significa nada.